# 第八课｜我们怎么知道硬件是对的？

上一课已经有一块可读的 **寄存器传输级（Register-Transfer Level, RTL）** neuron。代码“看起来合理”还不够。今天只解决：
> **没有真实 FPGA 板时，怎样给 RTL 输入、观察输出，并让错误自动失败？**

主要新概念：**硬件仿真（simulation）与测试平台（testbench）**。


## 1. 概念账本

**已经知道：** module/port、combinational path、clocked register update。

**今天学习：** simulation、testbench、waveform、Device Under Test (DUT)。

**只预告：** synthesis 与真实 FPGA 会在后面的平台课程中单独学习。


## 2. 三个词

**simulation**：软件执行 HDL 的逻辑/时序语义，预测给定 stimulus 下的行为；它不是实体芯片。

**testbench**：只为验证而写的 HDL，负责产生 clock/reset/input 并检查输出。

**waveform**：把数字信号随仿真时间的变化画出来。被测模块常简称 **DUT（Device Under Test）**。


## 3. testbench 与 design RTL 要分开

testbench 可以使用延迟、打印、`$fatal` 和 `$finish`；这些不是要综合进 FPGA 的逻辑。

本 testbench 顶部的 `` `timescale 1ns/1ps `` 表示时间单位是 1 ns、仿真精度是 1 ps。`#4`、`#1`、`#5` 都是 testbench 时间推进。


## 4. 先写 oracle，再看 waveform

先由已知 contract 得到 expected values：


In [ ]:
state = 0
threshold = 4
print('cycle | input | before | candidate | spike | after')
for cycle, current in enumerate([1, 1, 1, 1, 2, 2]):
    before = state
    candidate = before + current
    spike = candidate >= threshold
    state = 0 if spike else candidate
    print(f'{cycle:5d} | {current:5d} | {before:6d} | {candidate:9d} | {int(spike):5d} | {state:5d}')


## 5. DUT 端口要显式连接

本课故意不用 `dut(.*)`。显式写 `.clk(clk)`、`.threshold(threshold)` 等连接，学生可以直接核对“哪个 testbench signal 接到哪个 design port”。


## 6. self-checking testbench

testbench 在每个 edge 后比较 `membrane_v` 与 `spike`；不匹配立即 `$fatal`，全部通过才打印 `PASS lesson08 tutorial_if_neuron`。

`task automatic apply_and_check(...)` 只是 testbench 内部用来避免重复代码的小助手，不是 neuron design 的一部分。


## 7. 命令行到底做了什么？

```bash
mkdir -p build/lesson08
iverilog -g2012 -o build/lesson08/tutorial_if_neuron.vvp \
  rtl/learning/tutorial_if_neuron.sv \
  tb/learning/tutorial_if_neuron_tb.sv
cd build/lesson08 && vvp tutorial_if_neuron.vvp
```

第一条工具把 SystemVerilog 编译成仿真文件，第二条实际运行仿真。


## 8. Run：实际 simulation（如果 simulator 已安装）

Notebook 会把仿真产物保存在 `build/lesson08/`，不会再放到运行结束就删除的临时目录。


In [ ]:
from pathlib import Path
import shutil, subprocess

def repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p/'pyproject.toml').exists() and (p/'lessons').exists():
            return p
    raise FileNotFoundError('Run inside FPGA-FlyBrain')

root = repo_root()
iverilog = shutil.which('iverilog')
vvp = shutil.which('vvp')
if not (iverilog and vvp):
    print('Icarus Verilog not found; RTL simulation did not run.')
else:
    build = root/'build'/'lesson08'
    build.mkdir(parents=True, exist_ok=True)
    sim = build/'tutorial_if_neuron.vvp'
    subprocess.run([
        iverilog, '-g2012', '-o', str(sim),
        str(root/'rtl/learning/tutorial_if_neuron.sv'),
        str(root/'tb/learning/tutorial_if_neuron_tb.sv')
    ], check=True, cwd=build)
    result = subprocess.run([vvp, str(sim)], check=True, text=True, capture_output=True, cwd=build)
    print(result.stdout.strip())
    print('Waveform saved at:', build/'tutorial_if_neuron.vcd')


## 9. Waveform / VCD

testbench 用 `$dumpfile/$dumpvars` 生成 **Value Change Dump (VCD)** 波形文件：`build/lesson08/tutorial_if_neuron.vcd`。

有 GTKWave、Surfer 等 viewer 时可以打开它。没有 viewer 也不影响 self-checking testbench 的 pass/fail。


## 10. 失败时先查哪一层？

1. contract/oracle 是否正确；
2. testbench 是否在正确时间 drive / sample；
3. RTL combinational path / register update 是否错误。

不要让 AI 随机改 RTL 直到绿色出现。


## 11. Try It：故意制造可检测的 bug

在临时副本把 `>=` 改成 `>`。先预测哪个 boundary case 会失败，再运行 testbench，最后恢复原代码。


## 12. AI Task

给 AI 一条具体 failure message，让它分别从 oracle、testbench timing、RTL 三层提出检查点，并要求先引用证据再建议修改。


## 13. Human Check

不用 AI，你应该能解释 simulation 与真实 FPGA 的区别；testbench 为什么不是 design RTL；self-checking testbench 与 waveform 各自擅长什么；为什么 Python oracle 与 RTL 不一致时不能天然宣布 Python 一定正确。


## 14. Engineering Handoff

本课建立 RMD-005 / RMD-005A 的验证习惯。正式 LIF RTL 建立后，要用 Python fixed-point vectors 作为直接 oracle；tutorial testbench 不替代正式验证。


## 15. 项目追踪 Project Trace

- Lesson: `LSN-008`
- Mapping: `RMD-005 / RMD-005A`
- Prepared verification level: `L3 RTL unit simulation`


## 16. Exit Ticket

你能区分 compile 与 simulation、design RTL 与 testbench、self-checking assertion 与 waveform；并且不会把“simulation passed”说成“已经在 FPGA 上运行”。
